In [ ]:
%pip install pandas sqlalchemy ipython-sql jupysql matplotlib
%pip install "prettytable>=3.12.0"

In [ ]:
import sys
print(sys.version)
print(sys.executable)

# Intermediate Demo: One-Month Sari-Sari Store Simulator with Analytics

This notebook demonstrates the **Intermediate** goal of the Sari-Sari Store Simulator.

It exercises every Intermediate requirement from the project brief:

- Calculate revenues and expenses from a month's worth of transactions
- Produce inventory files (before sales and after-sales / restock plan)
- Generate a monthly dashboard and analytics report
- Automatically generate synthetic monthly transactions, parameterised by
  Philippine sari-sari store benchmarks (payday, weekend, category multipliers)

The implementation lives under `src/intermediate/`. This notebook is for
demonstration, validation, SQL Magic inspection, and visualisations.

## 1. Set up project paths

The notebook lives in `notebooks/`, so we point Python back to the project root
to enable `src.*` imports.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.intermediate.monthly_simulator import (
    DATABASE_PATH,
    INTERMEDIATE_OUTPUT_DIR,
    RAW_INVENTORY_PATH,
    SQL_TABLE_NAMES,
)

print("Inventory path     :", RAW_INVENTORY_PATH)
print("Output folder      :", INTERMEDIATE_OUTPUT_DIR)
print("Database path      :", DATABASE_PATH)
print("Inventory exists   :", RAW_INVENTORY_PATH.exists())

## 2. Preview the inventory

The Intermediate level uses the same `data/raw/inventory.csv` master list as
Basic and Advanced. It is never modified — outputs go to
`data/processed/intermediate/`.

In [ ]:
import pandas as pd

inventory_raw = pd.read_csv(RAW_INVENTORY_PATH)
print(f"Products  : {len(inventory_raw)}")
print(f"Categories: {sorted(inventory_raw['category'].unique())}")
display(inventory_raw)

## 3. Run the full Intermediate pipeline

`run_intermediate_monthly_simulator` performs all Intermediate steps in one
call:

1. Load the raw inventory and use it as the start-of-month snapshot
2. Auto-generate one month of synthetic transactions using PH sari-sari
   benchmark info (payday, weekend, category multipliers)
3. Compute transaction-level revenue, expense, and gross profit
4. Build the monthly product summary and one-row ledger
5. Produce the inventory-after-sales / restock recommendation file
6. Build the normalised monthly dashboard table
7. Save all eight outputs to CSV and SQLite, and emit dashboard PNGs

The benchmark dictionary below mirrors the assumptions used in the
Intermediate sanity-check suite (`test/intermediate/`).

In [ ]:
from src.intermediate.monthly_simulator import run_intermediate_monthly_simulator

benchmark_info = {
    "average_daily_customers": 50,
    "weekend_multiplier": 1.20,
    "payday_multiplier": 1.40,
    "category_multipliers": {
        "beverage": 1.35,
        "snacks":   1.30,
        "food":     1.20,
    },
}

outputs = run_intermediate_monthly_simulator(
    month="2026-01",
    benchmark_info=benchmark_info,
    random_seed=512,
    save_outputs=True,
    create_charts=True,
)

print("Outputs produced:")
for name, df in outputs.items():
    print(f"  {name:<35} rows={len(df):>5}")

In [ ]:
transactions          = outputs["monthly_transactions"]
inventory_before      = outputs["inventory_before_monthly_sales"]
transaction_details   = outputs["monthly_transaction_details"]
product_summary       = outputs["monthly_product_summary"]
ledger_summary        = outputs["monthly_ledger_summary"]
inventory_after_sales = outputs["inventory_after_monthly_sales"]
restock               = outputs["restock_recommendations"]
dashboard             = outputs["monthly_dashboard_data"]

## 4. Check the generated CSV files

All Intermediate outputs are saved to `data/processed/intermediate/`.

In [ ]:
expected_files = [
    "monthly_transactions.csv",
    "inventory_before_monthly_sales.csv",
    "monthly_transaction_details.csv",
    "monthly_product_summary.csv",
    "monthly_ledger_summary.csv",
    "inventory_after_monthly_sales.csv",
    "restock_recommendations.csv",
    "monthly_dashboard_data.csv",
]

print(f"Output folder: {INTERMEDIATE_OUTPUT_DIR}\n")
for fname in expected_files:
    path = INTERMEDIATE_OUTPUT_DIR / fname
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {fname}")

## 5. Inspect the synthetic transactions

The auto-generator produces a realistic stream of daily transactions across
all 15 products. Daily volume is shaped by the benchmark multipliers.

In [ ]:
print("Transactions shape :", transactions.shape)
print("Columns            :", list(transactions.columns))
display(transactions.head(8))

In [ ]:
import matplotlib.pyplot as plt

transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
daily_units = (
    transactions.groupby(transactions["transaction_date"].dt.date)["quantity_sold"]
    .sum()
    .reset_index(name="units")
)

plt.figure(figsize=(12, 3.5))
plt.plot(daily_units["transaction_date"], daily_units["units"], marker="o")
plt.title("Daily units sold — January 2026 (payday + weekend lift visible)")
plt.xlabel("Date")
plt.ylabel("Units sold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Transaction details, product summary, and one-row ledger

These three tables are the canonical Intermediate financial outputs.

In [ ]:
print("Transaction details shape:", transaction_details.shape)
display(transaction_details.head(5))

In [ ]:
print("Product summary shape:", product_summary.shape)
display(product_summary)

In [ ]:
print("Ledger summary (one row per month):")
display(ledger_summary)

## 7. Inventory before sales and restock plan for next month

Per the project brief, the Intermediate level produces:

- An inventory file for the month **prior to sales** (opening snapshot)
- An inventory file with the **restock plan for the next month**

In [ ]:
print("Inventory before monthly sales:")
display(inventory_before)

In [ ]:
print("Restock recommendations for next month:")
display(
    restock[[
        "product_id", "product_name", "category",
        "remaining_stock", "reorder_point",
        "recommend_restock", "recommended_restock_quantity",
        "stockout_risk", "restock_reason",
    ]]
)

n_restock = int(restock["recommend_restock"].astype(bool).sum())
print(f"\nProducts flagged for restock: {n_restock}")

## 8. Monthly dashboard and analytics report

The dashboard is a normalised `metric_group / metric_name / dimension / value`
table that downstream BI tools (or SQL Magic) can query directly.

In [ ]:
print("Dashboard shape:", dashboard.shape)
print("Metric groups  :", sorted(dashboard["metric_group"].unique()))
display(dashboard.head(10))

In [ ]:
kpis = dashboard[dashboard["metric_group"] == "kpi"][["metric_name", "value"]]
print("Monthly KPI summary:")
display(kpis)

In [ ]:
trend = dashboard[dashboard["metric_group"] == "daily_revenue_trend"].copy()
trend["value"] = pd.to_numeric(trend["value"])

plt.figure(figsize=(12, 3.5))
plt.plot(trend["dimension"], trend["value"], marker="o")
plt.title("Daily revenue trend — January 2026")
plt.xlabel("Date")
plt.ylabel("Revenue (PHP)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
cat = dashboard[dashboard["metric_group"] == "sales_by_category"].copy()
cat["value"] = pd.to_numeric(cat["value"])

plt.figure(figsize=(7, 4))
plt.bar(cat["dimension"], cat["value"])
plt.title("Sales by category — January 2026")
plt.xlabel("Category")
plt.ylabel("Revenue (PHP)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 9. Connect to SQLite using SQL Magic

All Intermediate outputs are also written to the shared
`src/database/sari_sari_store.db` file under tables prefixed `intermediate_*`.

In [ ]:
%load_ext sql
%sql sqlite:///../src/database/sari_sari_store.db

In [ ]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table'
  AND name LIKE 'intermediate_%'
ORDER BY name;

In [ ]:
%%sql
SELECT *
FROM intermediate_monthly_ledger_summary;

In [ ]:
%%sql
SELECT product_id, product_name, category,
       remaining_stock, reorder_point,
       recommend_restock, recommended_restock_quantity, stockout_risk
FROM intermediate_restock_recommendations
ORDER BY recommend_restock DESC, recommended_restock_quantity DESC;

In [ ]:
%%sql
SELECT product_id, product_name, total_quantity_sold,
       total_revenue, gross_profit
FROM intermediate_monthly_product_summary
ORDER BY total_quantity_sold DESC
LIMIT 10;

## 10. SQL ledger reconciliation

Recompute the month totals from `intermediate_monthly_transaction_details`
and confirm they match the stored ledger summary row.

In [ ]:
%%sql
SELECT ROUND(SUM(revenue),       2) AS total_revenue_calc,
       ROUND(SUM(expense),       2) AS total_expense_calc,
       ROUND(SUM(gross_profit),  2) AS total_profit_calc
FROM intermediate_monthly_transaction_details;

In [ ]:
%%sql
SELECT total_revenue, total_expense, gross_profit
FROM intermediate_monthly_ledger_summary;

## 11. Final Intermediate goal validation

In [ ]:
details_total = round(float(transaction_details["revenue"].sum()),       2)
ledger_total  = round(float(ledger_summary.iloc[0]["total_revenue"]),    2)
remaining_ok  = (
    (product_summary["starting_stock"] - product_summary["total_quantity_sold"])
    .astype(int)
    .equals(product_summary["remaining_stock"].astype(int))
)

checks = {
    "month_of_transactions_loaded":       len(transactions) > 0,
    "synthetic_data_generation_ran":      len(transactions) > 0,
    "benchmark_info_applied":             True,  # passed via run_* call above
    "inventory_before_file_produced":     (INTERMEDIATE_OUTPUT_DIR / "inventory_before_monthly_sales.csv").exists(),
    "inventory_after_file_produced":      (INTERMEDIATE_OUTPUT_DIR / "inventory_after_monthly_sales.csv").exists(),
    "restock_plan_produced":              len(restock) > 0,
    "monthly_dashboard_built":            len(dashboard) > 0,
    "ledger_matches_details":             details_total == ledger_total,
    "remaining_stock_equation_holds":     remaining_ok,
    "sqlite_db_present":                  DATABASE_PATH.exists(),
}

print("Intermediate goal validation:\n")
all_passed = True
for name, ok in checks.items():
    flag = "PASS" if ok else "FAIL"
    if not ok:
        all_passed = False
    print(f"  [{flag}] {name}")

print()
print(
    "Intermediate goal completed successfully."
    if all_passed else
    "Some checks failed — review the output above."
)

In [ ]:
# Cleanup: close all open database connections.
# This releases the lock on sari_sari_store.db so the next notebook
# can write to it without needing a kernel restart.
try:
    from sql.connection import ConnectionManager
    ConnectionManager.close_all()
    print("All SQL connections closed.")
except Exception as e:
    print(f"Skipped (SQL magic not loaded): {e}")


## End of Intermediate Demo

This notebook exercised the Intermediate level of the Sari-Sari Store
Simulator. The implementation lives under `src/intermediate/`, with the
matching sanity-check suite in `test/intermediate/`.